# Video Game Sales & Engagement Analysis - PHASE 2

In [41]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import pyodbc
import urllib



IMPORT THE CLEAN DATASET Engagement

In [24]:
df_CleanEngagementData = pd.read_csv('CleanEngagementData.csv')
df_CleanEngagementData.head(2)

,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,Title_norm
0,Elden Ring,2022-02-25,"['Bandai Namco Entertainment', 'FromSoftware']",4.5,3900.0,3900.0,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17000.0,3800.0,4600.0,4800.0,elden ring
1,Hades,2019-12-10,['Supergiant Games'],4.3,2900.0,2900.0,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21000.0,3200.0,6300.0,3600.0,hades


# Create the Dataset to Star Schema (The Gold Standard) : PascalCase 

In [25]:
# Creating Reusable function for PascalCase conversion

import re 

def to_pascal_case(column_name):  
    column_name = column_name.replace('_', ' ') # Replace underscores with spaces
    column_name = re.sub(r'[^\w\s]', '', column_name)    # Remove special characters except spaces
    words = column_name.split()

    pascal_words = []
    
    for word in words:
        if word.isupper():     # If fully uppercase (like NA, EU, JP)
            pascal_words.append(word.title())
        else:
            pascal_words.append(word[0].upper() + word[1:])
    
    return ''.join(pascal_words)



In [26]:
# Apply to Entire DataFrame
df_CleanEngagementData.columns = [to_pascal_case(col) for col in df_CleanEngagementData.columns]

In [27]:
df_CleanEngagementData.head(2)

,Title,ReleaseDate,Team,Rating,TimesListed,NumberOfReviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,TitleNorm
0,Elden Ring,2022-02-25,"['Bandai Namco Entertainment', 'FromSoftware']",4.5,3900.0,3900.0,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17000.0,3800.0,4600.0,4800.0,elden ring
1,Hades,2019-12-10,['Supergiant Games'],4.3,2900.0,2900.0,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21000.0,3200.0,6300.0,3600.0,hades


# IMPORT THE CLEAN DATASET - Sales

In [28]:
df_CleanSalesData = pd.read_csv('CleanSalesData.csv')
df_CleanSalesData.head(2)

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Name_norm
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,wii sports
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,super mario bros


In [29]:
df_CleanSalesData.columns = [to_pascal_case(col) for col in df_CleanSalesData.columns]
df_CleanSalesData.head(2)

,Rank,Name,Platform,Year,Genre,Publisher,NaSales,EuSales,JpSales,OtherSales,GlobalSales,NameNorm
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,wii sports
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,super mario bros


# Create DimGame (Master Game Table)

In [30]:
DimGame = pd.DataFrame(
    {
        'GameName': list(
            set(df_CleanEngagementData['TitleNorm'])
            .union(set(df_CleanSalesData['NameNorm']))
        )
    }
)


In [31]:
DimGame = DimGame.sort_values('GameName').reset_index(drop=True) 

In [ ]:
# Surrogate Key 
DimGame['GameId'] = range(1,len(DimGame)+1) # Warehouse foreign key.

In [33]:
DimGame

,GameName,GameId
0,007 quantum of solace,1
1,007 racing,2
2,007 the world is not enough,3
3,007 tomorrow never dies,4
4,1 vs 100,5
...,...,...
12074,zumba fitness rush,12075
12075,zumba fitness world party,12076
12076,zwei,12077
12077,zyuden sentai kyoryuger game de gaburincho,12078


# MERGING DATASETS (STAGING TABLES)

In [35]:
# Merge engagement dataset to DimGame

df_CleanEngagementData = df_CleanEngagementData.merge(
    DimGame,
    left_on='TitleNorm',
    right_on='GameName',
    how='left'
)

In [37]:
df_CleanEngagementData.head(2)

,Title,ReleaseDate,Team,Rating,TimesListed,NumberOfReviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,TitleNorm,GameName,GameId
0,Elden Ring,2022-02-25,"['Bandai Namco Entertainment', 'FromSoftware']",4.5,3900.0,3900.0,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17000.0,3800.0,4600.0,4800.0,elden ring,elden ring,2841
1,Hades,2019-12-10,['Supergiant Games'],4.3,2900.0,2900.0,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21000.0,3200.0,6300.0,3600.0,hades,hades,4063


In [36]:
# Merge Sales dataset to DimGame
df_CleanSalesData = df_CleanSalesData.merge(
    DimGame,
    left_on='NameNorm',
    right_on='GameName',
    how='left'
)

In [38]:
df_CleanSalesData.head(2)

,Rank,Name,Platform,Year,Genre,Publisher,NaSales,EuSales,JpSales,OtherSales,GlobalSales,NameNorm,GameName,GameId
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,wii sports,wii sports,11556
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,super mario bros,super mario bros,9815


# SQL CONNECTION 

In [ ]:
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 18 for SQL Server']


In [42]:
params = urllib.parse.quote_plus(
    "DRIVER=ODBC Driver 18 for SQL Server;"
    "SERVER=ANIRUDH\\SQLEXPRESS;"
    "DATABASE=VideoGameAnalysis;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

# TEST CONNECTION

In [43]:
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT @@SERVERNAME AS server_name, DB_NAME() AS database_name")
    )
    print(result.fetchone())

('Anirudh\\SQLEXPRESS', 'VideoGameAnalysis')


# LOAD DATA TO SQL

In [44]:
df_CleanEngagementData.to_sql(
    'df_CleanEngagementData',
    engine,
    if_exists='replace',
    index='False',
    chunksize=1000
)

df_CleanSalesData.to_sql(
    'df_CleanSalesData',
    engine,
    if_exists='replace',
    index='False',
    chunksize=1000
)

-17